# VinaCare · modelo de exercícios (OCI Data Science)

Este notebook treina o modelo que olha os pontos do corpo (MediaPipe Pose) e diz **qual exercício** a pessoa está fazendo e **se está certo ou errado**.

Caminho: **servidor VinaCare → este notebook (normalizar, ângulos, treinar, avaliar) → Model Catalog → Model Deployment**.

Rode as células em ordem (Shift+Enter). Kernel: *General Machine Learning for CPUs on Python 3.11* (conda `generalml_p311_cpu_x86_64_v1`).

## 1. Configuração

In [ ]:
import getpass, json, collections
import numpy as np, pandas as pd, requests

API = "https://vinacare.sandlj.com.br"
CODIGO = getpass.getpass("Código de coleta: ")

# Baixa do GitHub o MESMO cálculo de características que o back-end usa
url = "https://raw.githubusercontent.com/LeonardoGMendoza/VinaCare-AI/main/backend/app/features.py"
open("features.py", "w", encoding="utf-8").write(requests.get(url, timeout=30).text)
import importlib, features; importlib.reload(features)
print("features.py ok:", len(features.nomes()), "características por gravação")

## 2. Baixar as gravações do servidor

In [ ]:
r = requests.get(API + "/api/dataset/exportar", headers={"x-coleta-codigo": CODIGO}, timeout=60)
r.raise_for_status()
gravacoes = r.json()
json.dump(gravacoes, open("dataset_vinacare.json", "w"))
print(len(gravacoes), "gravações baixadas")
pd.crosstab(pd.Series([g["exercicio"] for g in gravacoes], name="exercício"),
            pd.Series([g["qualidade"] for g in gravacoes], name="qualidade"))

## 3. Limpar e transformar em números

Cada gravação vira um vetor fixo: o corpo é **centralizado no quadril** e **medido em "troncos"** (assim não importa a altura da pessoa nem onde ela está na tela). Depois calculamos os **ângulos** de ombro, cotovelo, quadril e joelho e a posição dos pulsos, quadro a quadro, e resumimos cada curva (mínimo, máximo, amplitude, velocidade…).

Gravações em que a câmera não pegou o corpo (visibilidade < 0,5) são descartadas.

In [ ]:
VIS_MIN = 0.5
X, y, pessoas, descartadas = [], [], [], 0
for g in gravacoes:
    if features.visibilidade(g["quadros"]) < VIS_MIN:
        descartadas += 1
        continue
    X.append(features.caracteristicas(g["quadros"], g.get("fps", 30)))
    y.append(g["exercicio"] + "|" + g["qualidade"])
    pessoas.append(g.get("pessoa", "anonimo"))
X, y, pessoas = np.array(X), np.array(y), np.array(pessoas)
print("Usadas:", len(X), "· descartadas (corpo fora da câmera):", descartadas)
print("Pessoas diferentes:", len(set(pessoas)))
pd.Series(y).value_counts().sort_index()

## 4. Treinar e avaliar

Modelo: **Random Forest** (funciona bem com poucos dados e mostra quais medidas pesaram mais).

Avaliação com **validação cruzada**: o modelo é testado em gravações que ele não viu no treino. Se houver mais de uma pessoa, também testamos **em uma pessoa que ele nunca viu** — essa é a nota que vale para o mundo real.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def novo_modelo():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)

contagem = collections.Counter(y)
k = min(5, min(contagem.values()))
if len(contagem) < 2 or k < 2:
    print("Ainda poucos dados: preciso de pelo menos 2 classes com 2+ gravações cada.", dict(contagem))
else:
    prev = cross_val_predict(novo_modelo(), X, y, cv=StratifiedKFold(k, shuffle=True, random_state=42))
    print(f"Acurácia (validação cruzada, {k} partes): {accuracy_score(y, prev):.1%}\n")
    print(classification_report(y, prev, zero_division=0))
    rotulos = sorted(contagem)
    display(pd.DataFrame(confusion_matrix(y, prev, labels=rotulos), index=rotulos, columns=rotulos))

    if len(set(pessoas)) >= 2:
        g = min(5, len(set(pessoas)))
        prev_p = cross_val_predict(novo_modelo(), X, y, groups=pessoas, cv=GroupKFold(g))
        print(f"Acurácia em pessoa nunca vista ({g} grupos): {accuracy_score(y, prev_p):.1%}")
    else:
        print("Só uma pessoa gravou: grave com outras pessoas para medir se o modelo generaliza.")

## 5. Treino final e o que o modelo aprendeu

In [ ]:
modelo = novo_modelo().fit(X, y)
imp = pd.Series(modelo.feature_importances_, index=features.nomes()).sort_values(ascending=False)
imp.head(15).plot.barh(figsize=(7, 5), title="Medidas que mais pesaram").invert_yaxis()
imp.head(15)

In [ ]:
import joblib
joblib.dump(modelo, "vinacare_exercicios.joblib")
print("Modelo salvo em vinacare_exercicios.joblib")

## 6. Publicar: Model Catalog + Model Deployment

Usa a biblioteca **ADS** da Oracle (já vem no kernel). O notebook se autentica como *resource principal* — isso precisa da política do passo 0 do guia (dynamic group do Data Science).

⚠️ O Model Deployment fica **ligado e cobrando** enquanto estiver ativo. Depois da demonstração, desative em *Data Science → Model deployments → Deactivate*.

In [ ]:
import ads
from ads.model.framework.sklearn_model import SklearnModel
from ads.common.model_metadata import UseCaseType
ads.set_auth("resource_principal")

publicado = SklearnModel(estimator=modelo, artifact_dir="./artefato_vinacare")
publicado.prepare(
    inference_conda_env="generalml_p311_cpu_x86_64_v1",
    inference_python_version="3.11",
    X_sample=X[:3], y_sample=y[:3],
    use_case_type=UseCaseType.MULTINOMIAL_CLASSIFICATION,
    force_overwrite=True,
)
print(publicado.verify(X[:2].tolist()))   # testa o score.py localmente

In [ ]:
id_modelo = publicado.save(display_name="vinacare-exercicios", description="VinaCare: exercício + certo/errado a partir do MediaPipe Pose")
print("Model Catalog:", id_modelo)

In [ ]:
publicado.deploy(
    display_name="vinacare-exercicios",
    deployment_instance_shape="VM.Standard.E4.Flex",
    deployment_ocpus=1,
    deployment_memory_in_gbs=8,
)
print("Endpoint:", publicado.model_deployment.url)

In [ ]:
# Teste do endpoint publicado com uma gravação real
print(publicado.predict(X[:1].tolist()))
print("Esperado:", y[0])